In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.4 Laplace's and Poisson's Equations

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume III — Classical Electrodynamics",
    number="3.4",
    title="Laplace's and Poisson's Equations",
    blurb="The boundary-value problem and how to solve it: relaxation methods "
    "(Jacobi, Gauss–Seidel, SOR) and their convergence, the sparse "
    "linear-system view, sources via Poisson, and the method of images. "
    "The computational heart of the volume — long, and worth it.",
    difficulty="advanced",
    estimate="150–200 min",
)

## Notebook overview

A frank word before we start: this is a long notebook, and deliberately so. The
boundary-value problem and the solvers we build for it are the computational core
that the rest of this volume, and a surprising amount of the rest of physics, rests
on. We would rather develop it once, carefully and whole, than scatter it thinly.
It earns its length, and it is, we think, the most rewarding chapter in the volume.

Here the question of electrostatics turns inside out.
[§3.1](coulomb-field.ipynb)–[§3.3](gauss-law.ipynb) worked in
one direction: *given the charges, sum or integrate for the field.* But most real
problems hand us the opposite. We are told the **boundaries** (the potentials on
conductors, the voltages on walls) and asked for the field that fills the space
between them. That is the **boundary-value problem**, and summing Coulomb's law
cannot touch it: we do not know where the charges (induced on the conductors) even
are. We need genuinely new machinery, and we build it.

The notebook runs in four movements. **I — discretization and relaxation:** the
five-point Laplacian, and solving Laplace's equation by iterated neighbour-averaging
(Jacobi), watched as it happens. **II — convergence and the linear-system view:**
why Gauss–Seidel beats Jacobi, how successive over-relaxation (SOR) beats both at the
right $\omega$, and the recognition that the whole thing is one big sparse linear
system that a direct solver can crack too. **III — sources:** Poisson's equation,
fields with no closed form. **IV — the method of images:** the elegant trick that
turns special geometries into a few fictitious charges, cross-checked against
relaxation. We close on the realisation that $\nabla^2\varphi = \text{source}$ is
the equation of equilibrium *everywhere* in physics, not just here.

Everything is in **SI units** ($\varepsilon_0 = 8.854\times10^{-12}\,$F/m). We work
in **2-D** throughout, where the fields and the convergence are easy to see and to
animate; the methods generalise to 3-D essentially unchanged (only the pictures get
harder), which we note where it matters.

> **How to read the checks.** Each exercise ends with a `validate` call against an
> independent fact: a discrete Laplacian that vanishes, a relaxed field that matches
> an analytic series, a direct solve that matches relaxation, an image construction
> that zeroes the potential on a conductor. A ✓ is strong evidence; a ✗ is a prompt
> to *locate the discrepancy*, not a verdict.

> **Scope.** A working review, not a numerical-analysis course. See Nolting,
> *Theoretical Physics 3* {cite}`nolting3`; Griffiths, *Introduction to
> Electrodynamics* {cite}`griffiths_em` (ch. 3); Press et al., *Numerical Recipes*
> {cite}`numrecipes` (ch. 20, relaxation/PDEs); and Trefethen & Bau
> {cite}`trefethen1997` for the iterative-methods/spectral-radius view.

## Theory in brief

### Laplace, Poisson, and uniqueness

Combining $\mathbf E = -\nabla\varphi$ ([§3.2](potential-energy.ipynb)) with Gauss's
law $\nabla\cdot\mathbf E = \rho/\varepsilon_0$ ([§3.3](gauss-law.ipynb)) gives the
equation the potential obeys,

```{math}
:label: eq-lp-poisson
\nabla^2\varphi = -\frac{\rho}{\varepsilon_0}\quad\text{(Poisson)},
\qquad
\nabla^2\varphi = 0\quad\text{(Laplace, where }\rho=0\text{)} .
```

The **uniqueness theorem** is what makes this useful: in a region where $\varphi$ is
fixed on the entire boundary (Dirichlet conditions), Poisson's equation has exactly
*one* solution. So however we find a solution that obeys the equation and matches the
boundary, it is guaranteed to be *the* solution. That licence (any valid solution is
the solution) is what lets a numerical guess-and-relax procedure be trusted.

### The mean-value property

A function with $\nabla^2\varphi=0$ is called **harmonic**, and harmonic functions
have a remarkable property (Griffiths {cite}`griffiths_em`, ch. 3, gives the short
proof): the value at any point equals the average of the values
on any sphere (in 2-D, circle) centred there,

```{math}
:label: eq-meanvalue
\varphi(\mathbf r) = \big\langle \varphi \big\rangle_{\text{sphere about }\mathbf r} .
```

An immediate corollary: a harmonic function can have **no local maximum or minimum
in the interior** (a peak would exceed its own surrounding average). The extremes
live on the boundary. This is Earnshaw's theorem again (callback to
[§3.2](potential-energy.ipynb): no stable
electrostatic trap), and it is the conceptual seed of everything below.

### Discretizing the Laplacian

On a grid of spacing $h$, a Taylor expansion gives the **five-point stencil**,

```{math}
:label: eq-stencil
\nabla^2\varphi \approx
\frac{\varphi_{i+1,j}+\varphi_{i-1,j}+\varphi_{i,j+1}+\varphi_{i,j-1}-4\varphi_{i,j}}
{h^2} .
```

Setting it to zero rearranges to $\varphi_{i,j} = \tfrac14(\varphi_{i+1,j}+
\varphi_{i-1,j}+\varphi_{i,j+1}+\varphi_{i,j-1})$: **each point is the average of its
four neighbours**, the mean-value property in discrete form.

### Relaxation as iterated averaging

That rearrangement is also an *algorithm*: sweep the grid, replace each interior point
by its neighbour-average (plus a source term for Poisson), and repeat until nothing
changes. The boundary values, held fixed, diffuse inward to the harmonic solution.
Three schemes differ only in *when* updated values are used:

- **Jacobi** {eq}`eq-lp-jacobi`: update every point from the values of the *previous*
  sweep.
  ```{math}
  :label: eq-lp-jacobi
  \varphi^{k+1}_{i,j} = \tfrac14\big(\varphi^{k}_{i+1,j}+\varphi^{k}_{i-1,j}
  +\varphi^{k}_{i,j+1}+\varphi^{k}_{i,j-1}\big) + \tfrac{h^2}{4}\,\frac{\rho_{i,j}}{\varepsilon_0}.
  ```
- **Gauss–Seidel** {eq}`eq-gs`: use neighbours *already updated* this sweep. Roughly
  twice as fast, and it needs no second copy of the grid.
  ```{math}
  :label: eq-gs
  \varphi^{k+1}_{i,j} = \tfrac14\big(\varphi^{k}_{i+1,j}+\varphi^{k+1}_{i-1,j}
  +\varphi^{k}_{i,j+1}+\varphi^{k+1}_{i,j-1}\big) + \tfrac{h^2}{4}\,\frac{\rho_{i,j}}{\varepsilon_0}.
  ```
- **SOR** (successive over-relaxation) {eq}`eq-sor`: take the Gauss–Seidel step and
  *overshoot* it by a factor $\omega\in(1,2)$. At the right $\omega$ this is
  dramatically faster.
  ```{math}
  :label: eq-sor
  \varphi^{k+1}_{i,j} = (1-\omega)\,\varphi^{k}_{i,j}
  + \omega\,\varphi^{\text{GS}}_{i,j}.
  ```

### Convergence: splittings and the spectral radius

Every such scheme is a linear iteration $\boldsymbol\varphi_{k+1} = M\boldsymbol
\varphi_k + \mathbf c$, and the error after each sweep is multiplied by the matrix
$M$. The error therefore shrinks by the **spectral radius** $\rho(M)$ (the largest
eigenvalue magnitude, callback to [§0.5](../00-foundations/eigenvalues-svd.ipynb))
per sweep,

```{math}
:label: eq-spectral-radius
\|\mathbf e_{k}\| \sim \rho(M)^{\,k}\,\|\mathbf e_0\| .
```

Jacobi, Gauss–Seidel, and SOR are three different **splittings** of the same system
matrix into "easy-to-invert plus remainder," giving three different $M$ and three
different $\rho(M)$, hence three convergence speeds. The smaller $\rho(M)$, the
faster. For the model problem on an $n\times n$ grid the optimal SOR parameter has a
closed form (Press et al., *Numerical Recipes* {cite}`numrecipes`, ch. 20, derives it
from the Jacobi spectral radius),

```{math}
:label: eq-omega-opt
\omega_{\text{opt}} = \frac{2}{1+\sin(\pi/n)} ,
```
which pushes $\rho(M)$ from $\approx 1-O(h^2)$ (Jacobi/G–S, painfully close to 1) down
to $\approx 1-O(h)$, the difference between $O(n^2)$ and $O(n)$ sweeps.

### The sparse linear-system view

The discrete equations {eq}`eq-stencil` are nothing but a large **linear system**,

```{math}
:label: eq-linear-system
A\,\boldsymbol\varphi = \mathbf b ,
```

with $A$ the (very sparse) five-point-stencil matrix and $\mathbf b$ holding the
boundary values and sources. Relaxation is an *iterative* solver for it; a **direct**
sparse solver (`scipy.sparse.linalg.spsolve`, an $LU$ factorization, callback to
[§0.4](../00-foundations/linear-systems.ipynb))
is the alternative. Direct is exact but memory-hungry; iterative is approximate but
scales. Same system, two philosophies.

### The method of images

For a few special boundary shapes the BVP can be solved *exactly* with a trick: replace
the conductor by fictitious **image charges** placed so the boundary condition falls
out by symmetry,

```{math}
:label: eq-images
\varphi(\mathbf r) = \frac{1}{4\pi\varepsilon_0}\sum_i \frac{q_i}{|\mathbf r-\mathbf r_i|},
\quad\text{real charges + images, chosen so }\varphi=0\text{ on the conductor.}
```

A charge $q$ at height $d$ above a grounded plane is, in the region above, equivalent to
$q$ plus an image $-q$ at $-d$. A charge $q$ a distance $d$ from the centre of a
grounded sphere of radius $a$ has the Kelvin image $q' = -qa/d$ at $b = a^2/d$. Images
give exact answers for special shapes; relaxation handles *any* shape. Where both apply,
they validate each other.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from matplotlib.animation import FuncAnimation

from ecp import draw, validate
from ecp.animate import show

from scipy.constants import epsilon_0 as EPS0  # vacuum permittivity, F/m

K = 1.0 / (4.0 * np.pi * EPS0)  # Coulomb constant ≈ 8.99e9 N·m²/C²
POS, NEG = "#c1121f", "#16213e"  # positive red, negative dark blue


def laplacian_5pt(phi, h):
    """The 5-point discrete Laplacian on a uniform grid (eq-stencil).

    (phi[i+1,j] + phi[i-1,j] + phi[i,j+1] + phi[i,j-1] - 4 phi[i,j]) / h^2,
    the finite-difference operator at the heart of the relaxation solvers;
    edges are left zero.

    Parameters
    ----------
    phi : numpy.ndarray
        The 2-D field.
    h : float
        Grid spacing.

    Returns
    -------
    numpy.ndarray
        The discrete Laplacian, same shape (interior only).
    """
    lap = np.zeros_like(phi)
    lap[1:-1, 1:-1] = (
        phi[2:, 1:-1]
        + phi[:-2, 1:-1]
        + phi[1:-1, 2:]
        + phi[1:-1, :-2]
        - 4.0 * phi[1:-1, 1:-1]
    ) / h**2
    return lap


def jacobi(phi, fixed, rhs, h, tol=1e-6, max_iter=20000, record=0):
    """Jacobi relaxation for a boundary-value Poisson/Laplace problem (eq-jacobi).

    Repeatedly replaces each interior node by the average of its neighbours
    (plus the source term), holding the boundary fixed, until the update falls
    below ``tol``.

    Parameters
    ----------
    phi : numpy.ndarray
        Initial field (boundary values already set).
    fixed : numpy.ndarray
        Boolean mask of held nodes.
    rhs : numpy.ndarray
        Source term ρ/ε0 (zero for Laplace); scaled by h^2/4 inside.
    h : float
        Grid spacing.
    tol : float, optional
        Convergence tolerance on the max update (default 1e-6).
    max_iter : int, optional
        Iteration cap (default 20000).
    record : int, optional
        If >0, also return every ``record``-th frame (default 0).

    Returns
    -------
    numpy.ndarray or tuple
        The relaxed field, plus a list of recorded frames when ``record`` > 0.
    """
    phi = phi.astype(float).copy()
    free = ~fixed
    hist, frames = [], []
    for it in range(1, max_iter + 1):
        nb = phi.copy()
        nb[1:-1, 1:-1] = (
            0.25 * (phi[2:, 1:-1] + phi[:-2, 1:-1] + phi[1:-1, 2:] + phi[1:-1, :-2])
            + 0.25 * h**2 * rhs[1:-1, 1:-1]
        )
        new = np.where(free, nb, phi)
        change = float(np.max(np.abs(new[free] - phi[free])))
        phi = new
        hist.append(change)
        if record and (it % record == 0):
            frames.append(phi.copy())
        if change < tol:
            break
    return phi, it, hist, frames


def sor(phi, fixed, rhs, h, omega, tol=1e-6, max_iter=20000):
    """Red-black successive over-relaxation (eq-sor); ``omega=1`` is Gauss-Seidel.

    Over-relaxes the Gauss-Seidel update by a factor ``omega`` in (1, 2),
    accelerating convergence; red-black ordering keeps the sweep vectorised.

    Parameters
    ----------
    phi : numpy.ndarray
        Initial field with boundary set.
    fixed : numpy.ndarray
        Boolean mask of held nodes.
    rhs : numpy.ndarray
        Source term.
    h : float
        Grid spacing.
    omega : float
        Over-relaxation factor.
    tol : float, optional
        Convergence tolerance (default 1e-6).
    max_iter : int, optional
        Iteration cap (default 20000).

    Returns
    -------
    tuple
        The relaxed field and the number of sweeps taken.
    """
    phi = phi.astype(float).copy()
    free = ~fixed
    I, J = np.indices(phi.shape)
    colours = [((I + J) % 2 == 0) & free, ((I + J) % 2 == 1) & free]
    hist = []
    for it in range(1, max_iter + 1):
        maxc = 0.0
        for m in colours:
            gs = (
                0.25
                * (
                    np.roll(phi, 1, 0)
                    + np.roll(phi, -1, 0)
                    + np.roll(phi, 1, 1)
                    + np.roll(phi, -1, 1)
                )
                + 0.25 * h**2 * rhs
            )
            upd = (1.0 - omega) * phi + omega * gs
            delta = np.where(m, upd - phi, 0.0)
            phi = phi + delta
            maxc = max(maxc, float(np.max(np.abs(delta))))
        hist.append(maxc)
        if maxc < tol:
            break
    return phi, it, hist


def box_bc(N, V_top=1.0):
    """Boundary conditions for the unit-square box problem.

    Three grounded edges and a top edge held at ``V_top``, the canonical Dirichlet test case.

    Parameters
    ----------
    N : int
        Grid points per side.
    V_top : float, optional
        Potential on the top edge (default 1.0).

    Returns
    -------
    phi, fixed : numpy.ndarray
        The initialised field and the boolean mask of held nodes.
    """
    x = np.linspace(0.0, 1.0, N)
    y = np.linspace(0.0, 1.0, N)
    h = x[1] - x[0]
    phi0 = np.zeros((N, N))
    phi0[:, -1] = V_top  # top edge y=1
    fixed = np.zeros((N, N), bool)
    fixed[0, :] = fixed[-1, :] = fixed[:, 0] = fixed[:, -1] = True
    return x, y, phi0, fixed, h


def box_series(x, y, V_top=1.0, n_terms=80):
    """Analytic Fourier-series solution of the box problem.

    The exact separable solution (a sum of sinh/sin modes), the truth the
    relaxed field is checked against.

    Parameters
    ----------
    x, y : numpy.ndarray
        Coordinates in the unit square.
    V_top : float, optional
        Top-edge potential (default 1.0).
    n_terms : int, optional
        Number of series terms (default 80).

    Returns
    -------
    numpy.ndarray
        The analytic potential at ``(x, y)``.
    """
    X, Y = np.meshgrid(x, y, indexing="ij")
    phi = np.zeros_like(X)
    for n in range(1, 2 * n_terms, 2):  # odd n only
        a = n * np.pi
        ratio = (
            np.exp(a * (Y - 1.0))
            * (1.0 - np.exp(-2.0 * a * Y))
            / (1.0 - np.exp(-2.0 * a))
        )
        phi += (4.0 * V_top / (n * np.pi)) * np.sin(a * X) * ratio
    return phi

# Movement I — Discretization and relaxation

## Exercise 1 — Discretizing the Laplacian

Everything starts with the five-point stencil {eq}`eq-stencil`: it turns the
differential operator $\nabla^2$ into a weighted sum over a point and its four
neighbours ({numref}`fig-lp-stencil`). The deepest sanity check is that it sends
*harmonic* functions to (nearly) zero, since those are exactly the functions
$\nabla^2$ annihilates. Two clean test functions are $u = x^2 - y^2$ and $u = xy$:
both satisfy $\nabla^2 u = 0$ exactly, and for these quadratics the central
difference is exact, so the discrete Laplacian should return zero to round-off.

**This exercise.** Implement `laplacian_5pt` (done in Setup) and apply it on a
$41\times41$ grid over $[-1,1]^2$ to $u=x^2-y^2$ and $u=xy$. Confirm the discrete
Laplacian is zero on the interior to machine precision, the discrete echo of
$\nabla^2 u = 0$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    max_lap,
    0.0,
    "the 5-point Laplacian of a harmonic function (x²−y², xy) is ≈ 0",
    atol=1e-10,
)

## Exercise 2 — Laplace by Jacobi relaxation

Now the central computation of the notebook. Take the unit square with three grounded
edges and the top edge held at $V_0 = 1\,$V ({numref}`fig-lp-box`). The interior
potential is unknown; we find it by **Jacobi relaxation** {eq}`eq-lp-jacobi`, sweeping
the grid and replacing every interior point by its neighbour-average until the largest
change in a sweep drops below a tolerance. The boundary, held fixed, diffuses inward.

This box is one of the rare shapes with an analytic answer, a Fourier series
$\varphi(x,y) = \sum_{n\text{ odd}} \frac{4V_0}{n\pi}\sin(n\pi x)\frac{\sinh(n\pi
y)}{\sinh(n\pi)}$, so we can grade relaxation against the truth. One honest caveat: the
**top corners** are boundary discontinuities (the top edge is at $V_0$, the sides at
$0$), and there the discrete grid and the truncated series disagree by $\sim V_0$, a
Gibbs-type effect of the singular corner. We therefore compare on the **interior**,
excluding the corner-adjacent nodes, and treat the corner mismatch as a lesson about
boundary singularities rather than a failure.

**This exercise.** On an $N=40$ grid:

1. Relax from a zero initial guess with **Jacobi** to tolerance $10^{-6}$
   (max change per sweep).
2. Compare the result to the analytic Fourier series (`box_series`) on the
   interior nodes.
3. Then *watch it happen*: animate the field relaxing sweep by sweep
   ({numref}`fig-lp-relax-anim`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    phi_jac[interior],
    phi_exact[interior],
    "Jacobi relaxation matches the analytic Fourier solution in the interior",
    atol=5e-3,
)

The animation below shows the *dynamics of convergence* a still cannot: the top
edge's value spreading down into the square, fast at first then ever more slowly as
the field settles onto the smooth harmonic solution.

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — The mean-value property and no interior extremum

The converged field is harmonic, and harmonic functions obey the **mean-value
property** {eq}`eq-meanvalue`: every interior value is the average of its neighbours.
A corollary is **no interior maximum or minimum** (Earnshaw again, callback to
[§3.2](potential-energy.ipynb)):
the hottest and coldest points sit on the boundary. Both are exact statements about
the continuous solution and near-exact about our discrete one.

**This exercise.** On the converged box solution:

1. Confirm each interior point equals its four-neighbour average (a
   vectorised sliced-array average) to the relaxation tolerance.
2. Confirm the interior maximum and minimum (`numpy.max`/`numpy.min`) are
   strictly bracketed by the boundary's ($0$ and $V_0$).

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    core,
    nbr_avg,
    "the converged potential satisfies the mean-value property",
    atol=1e-5,
)
validate.check(
    no_extremum,
    "no interior maximum or minimum: the extremes lie on the boundary (Earnshaw)",
)

# Movement II — Convergence and the linear-system view

## Exercise 4 — Gauss–Seidel versus Jacobi

Jacobi throws away information: within a sweep it keeps using last sweep's values even
after better ones are available. **Gauss–Seidel** {eq}`eq-gs` uses each updated
neighbour as soon as it is computed, which roughly halves the sweep count for the same
accuracy. We compare them on the identical box problem to the identical tolerance.

**This exercise.** Relax the $N=40$ box with Jacobi and with Gauss–Seidel
(`sor` at $\omega=1$) to tolerance $10^{-6}$, and count the sweeps each needs
({numref}`fig-lp-jac-gs`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    iters_gs < iters_jac,
    "Gauss–Seidel converges in fewer iterations than Jacobi",
    f"{iters_gs} vs {iters_jac} sweeps",
)

## Exercise 5 — SOR and the optimal $\omega$

The leap comes from **over-relaxation** {eq}`eq-sor`: take the Gauss–Seidel correction
and overshoot it by $\omega\in(1,2)$. The reason it helps is the spectral-radius story
{eq}`eq-spectral-radius`: each scheme is a splitting of $A\boldsymbol\varphi=\mathbf b$
into $M\boldsymbol\varphi_k+\mathbf c$, and the error dies as $\rho(M)^k$. For
Jacobi and Gauss–Seidel $\rho(M) = 1 - O(h^2)$ — agonizingly close to 1 on a fine
grid, so $O(n^2)$ sweeps. Over-relaxation tilts the splitting to *minimize* $\rho(M)$,
and at the optimum it drops to $1 - O(h)$, i.e. $O(n)$ sweeps. For the model problem
the minimizer is known in closed form, $\omega_{\text{opt}} = 2/(1+\sin(\pi/n))$
{eq}`eq-omega-opt`, and the iteration count has a sharp minimum there.

**This exercise.**

1. Sweep $\omega$ across $(1, 2)$, run SOR to tolerance $10^{-6}$ at each,
   and find the iteration-count minimum.
2. Compare the empirical best $\omega$ to $2/(1+\sin(\pi/(N-1)))$, and
   confirm SOR at the optimum is dramatically faster than Jacobi
   ({numref}`fig-lp-sor`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    omega_best,
    omega_opt,
    "the empirical optimal ω matches the closed form 2/(1+sin(π/(N-1)))",
    rtol=0.05,
)
validate.check(
    iters_sor < iters_jac / 5,
    "SOR at the optimal ω is dramatically faster than Jacobi",
    f"{iters_sor} vs {iters_jac} sweeps",
)

```{admonition} With your assistant
:class: tip
Relaxation solvers are a family, and your assistant knows more cousins than
this notebook shows — ask it for one (red–black ordering, a multigrid V-cycle,
whatever it offers) and run it on the boundary problem of Exercise 2. The
gates are already built: the analytic solution of that exercise, and the
mean-value property of Exercise 3, which any true harmonic solution must obey
regardless of who wrote the iteration. The check is yours.
```

## Exercise 6 — The sparse linear-system view

Step back and look at what relaxation has been solving. The discrete equations
{eq}`eq-stencil` are a single **linear system** $A\boldsymbol\varphi = \mathbf b$
{eq}`eq-linear-system`: one row per interior node, $-4$ on the diagonal and $+1$ for
each interior neighbour, with the known boundary values moved to the right-hand side
$\mathbf b$. The matrix is enormous but **sparse** (five non-zeros per row), so we
store it with `scipy.sparse` and solve it *directly* with
`scipy.sparse.linalg.spsolve` (a sparse $LU$ factorization, callback to
[§0.4](../00-foundations/linear-systems.ipynb)). The
relaxation methods are simply *iterative* solvers of this very system, with speed set
by the iteration matrix's spectral radius
([§0.5](../00-foundations/eigenvalues-svd.ipynb)).

**This exercise (student).**

1. Build the five-point-stencil matrix $A$ for the $N=40$ box as a
   `scipy.sparse` matrix, fold the Dirichlet boundary into $\mathbf b$, and
   solve $A\boldsymbol\varphi=\mathbf b$ with `scipy.sparse.linalg.spsolve`.
2. Confirm it matches the SOR relaxation on the interior.
3. Weigh the trade-off: the direct solve is exact and needs no tolerance,
   but factorization fills memory; relaxation is approximate but scales to
   grids a direct solve could never hold.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    phi_direct[interior],
    phi_sor[interior],
    "the direct sparse solve matches the relaxation solution",
    atol=1e-3,
)

# Movement III — Sources (Poisson)

## Exercise 7 — Poisson with a source

Add charge and Laplace becomes **Poisson** {eq}`eq-lp-poisson`, $\nabla^2\varphi =
-\rho/\varepsilon_0$. Relaxation barely changes: each averaging step simply gains the
source term $\tfrac14 h^2 \rho/\varepsilon_0$ {eq}`eq-lp-jacobi`. We place a smooth charge
blob inside a grounded box ({numref}`fig-lp-poisson-setup`), relax for $\varphi$, and
recover the field $\mathbf E = -\nabla\varphi$ by numerical gradient
([§3.2](potential-energy.ipynb)).

**This exercise.** In a grounded unit box ($N=80$, all edges $0\,$V), place a
Gaussian charge density
$\rho(x,y) = \rho_0\exp(-((x-\tfrac12)^2+(y-\tfrac12)^2)/2s^2)$ with
$s=0.06$ and a stated $\rho_0$.

1. Relax with **SOR** at $\omega_{\rm opt}$.
2. Verify the relaxed potential satisfies Poisson's equation on the interior
   (its 5-point Laplacian `laplacian_5pt` equals $-\rho/\varepsilon_0$).
3. Recover the field $\mathbf E=-\nabla\varphi$ with `numpy.gradient`,
   plotting it over the potential ({numref}`fig-lp-poisson-field`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    poisson_rel,
    0.0,
    "the relaxed potential satisfies Poisson's equation ∇²φ = −ρ/ε₀ in the interior",
    atol=2e-2,
)

## Exercise 8 — A field with no closed form

Here is the payoff. Put two electrodes at different potentials inside a grounded box
({numref}`fig-lp-electrodes-setup`) — a geometry with no series and no image solution.
Relaxation does not care: it solves *any* shape. We mark the electrode nodes as fixed
at their voltages and relax the charge-free region around them.

**This exercise.** In a grounded unit box ($N=80$), hold a small left
electrode at $+1\,$V and a small right electrode at $-1\,$V (stated
rectangular patches).

1. Relax (SOR at $\omega_{\rm opt}$) for the field between them.
2. With no closed form to compare, check the solution against the physics it
   must obey: in the charge-free region it is harmonic, so the **mean-value
   property** {eq}`eq-meanvalue` (a vectorised neighbour-average) must hold
   at every free interior node.
3. Recover the field $\mathbf E=-\nabla\varphi$ with `numpy.gradient`
   ({numref}`fig-lp-electrodes-field`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    mv_dev,
    0.0,
    "the numerical solution is harmonic (mean-value property) in the charge-free region",
    atol=1e-5,
)

# Movement IV — The method of images

## Exercise 9 — A charge above a grounded plane

For a few blessed geometries we can skip relaxation entirely. The classic is a charge
above a grounded plane ({numref}`fig-lp-image-plane`): the grounded boundary at $y=0$
can be reproduced *exactly* by deleting it and adding a fictitious **image** charge of
opposite sign at the mirror position {eq}`eq-images`. Working in the 2-D slice (a line
charge $\lambda$ at height $d$, image $-\lambda$ at $-d$), the potential is a sum of
two logarithms, and on the plane the two distances are equal, so $\varphi=0$ falls out
for free. Relaxation, given only the grounded boundary and the real charge, must
reproduce it.

**This exercise.** Place a line charge at $(0.5, d)$ with $d=0.3$ in the
upper half of a box whose bottom edge is grounded.

1. Build the image-charge potential (the sum of two logarithms) and verify
   it is zero on the grounded plane.
2. Relax the charge-free region with **SOR** (bottom grounded; the far edges
   and a small contour around the charge set to the analytic potential,
   excising the singularity) and confirm relaxation fills it with the same
   field the image construction predicts.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.close(
    max_on_plane,
    0.0,
    "the image construction gives zero potential on the grounded plane",
    atol=1e-12,
)
validate.close(
    phi_relax_i[free_i],
    phi_img[free_i],
    "relaxation reproduces the image-charge solution in the charge-free region",
    atol=5e-3,
)

## Exercise 10 — Induced charge and the force on the real charge

The image is fictitious, but the **induced surface charge** it stands in for is real.
For the 3-D point charge $q$ at height $d$ above a grounded plane, the induced density
is $\sigma(r) = -qd/\big(2\pi(r^2+d^2)^{3/2}\big)$ at distance $r$ from the foot of the
charge. Two famous results follow: the induced charge **totals exactly $-q$**, and the
attractive force on $q$ is exactly that of its image, $F = kq^2/(2d)^2$ (the charge is
pulled toward the plane as if a charge $-q$ sat a distance $2d$ away).

**This exercise (student).** With $q=1\,$nC and $d=0.05\,$m:

1. Integrate $\sigma(r)$ over the plane with `numpy.trapezoid` (in polar
   form, $\int_0^\infty \sigma(r)\,2\pi r\,dr$) and confirm it equals $-q$.
2. Compute the force $F=kq^2/(2d)^2$.

In [ ]:
# (solution hidden on the public site)


### Validation 10

In [ ]:
validate.close(
    induced_total,
    -q10,
    "the total induced surface charge equals −q",
    rtol=2e-3,
)
# independent cross-check: the image-force shortcut must equal the directly
# integrated vertical Coulomb pull of the *real* induced surface charge on q
force_from_sigma = float(
    np.trapezoid(
        K * q10 * np.abs(sigma) * d10 / (r**2 + d10**2) ** 1.5 * 2.0 * np.pi * r, r
    )
)
validate.close(
    force,
    force_from_sigma,
    "the image force kq²/(2d)² equals the integrated pull of the real induced charge",
    rtol=1e-3,
)

## Exercise 11 — The grounded sphere

The cleverest image is Kelvin's, for a **grounded sphere** of radius $a$
({numref}`fig-lp-image-sphere`). A charge $q$ a distance $d>a$ from the centre is
balanced, on the sphere, by a single image $q' = -qa/d$ placed *inside* at $b=a^2/d$
{eq}`eq-images`. The magic is that this one off-centre image makes the *entire curved
surface* an equipotential at zero, which is far from obvious (Griffiths
{cite}`griffiths_em`, ch. 3, works the construction through).

**This exercise.** With $a=0.1\,$m, $d=0.25\,$m, and $q=1\,$nC:

1. Construct the Kelvin image $q'=-qa/d$ at $b=a^2/d$.
2. Sample the sphere surface on a $(\theta,\varphi)$ grid (`numpy.meshgrid`)
   and evaluate the total potential (real charge + image) there, with
   point-to-charge distances from `numpy.linalg.norm`.
3. Confirm it is zero to round-off.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 11

In [ ]:
validate.close(
    max_on_sphere,
    0.0,
    "the Kelvin image q′=−qa/d at b=a²/d makes the sphere surface an equipotential at zero",
    atol=1e-12,
)

## Exercise 12 — The Laplacian everywhere

A closing widening of the lens. Nothing in Movements I–III was specific to
electrostatics. The equation $\nabla^2\varphi = \text{source}$, solved subject to
boundary conditions, is the equation of **equilibrium** across physics: it is
steady-state **heat conduction** (with $\varphi$ the temperature and the source a heat
input), **incompressible irrotational flow** (with $\varphi$ the velocity potential),
the shape of a stretched **soap film** (the minimal surface), and the smoothness of the
quantum ground state. The relaxation solver we built does not know or care which it is.

**This exercise.** Reinterpret the box problem of Exercise 2 as
**steady-state heat**: a square plate with three edges held at $0^\circ$ and
the top edge at $1^\circ$, at thermal equilibrium. The governing equation is
the *same* Laplace equation, so the *same* relaxation must give the *same*
field. Solve the "heat" problem with the same **SOR** relaxation and confirm
it reproduces the electrostatic solution exactly (same math, relabelled).

In [ ]:
# (solution hidden on the public site)


### Validation 12

In [ ]:
validate.close(
    T_steady[interior],
    phi_sor[interior],
    "the same relaxation solves the steady-state heat problem (same equation, relabelled)",
    atol=1e-6,
)

The Laplacian is the operator of equilibrium. Wherever a system settles into the
smoothest field consistent with what is fixed on its boundary — a potential between
conductors, a temperature across a plate, a flow around an obstacle — the same
$\nabla^2\varphi=\text{source}$ governs it, and the same relaxation and sparse-solver
machinery built here will crack it. That reach is why this notebook earned its length.

## Notebook summary

- Laplace's and Poisson's equations solved by **relaxation**: Jacobi and Gauss–Seidel
  iteration to a fixed boundary-value problem, with the animation showing $V$ diffusing
  inward and the change per sweep shrinking geometrically (spectral radius just below 1).
- The mean-value property and the no-interior-extremum theorem (Earnshaw) confirmed on the
  converged field; SOR accelerating convergence; and Poisson's equation with a source.
- The sparse linear-system view: the five-point matrix solved directly with
  `scipy.sparse.linalg.spsolve` and matching relaxation; a two-electrode field with no
  closed form trusted through the mean-value property; and the **method of images**
  (grounded plane, induced charge totalling $-q$, Kelvin's sphere) cross-validated
  against relaxation — closing with the Laplacian as the equation of equilibrium
  everywhere (the same solve, relabelled as steady-state heat).

## Outlook

- **Multigrid methods.** Relaxation kills short-wavelength error fast but long-wavelength
  error agonizingly slowly. Multigrid solves the problem on a hierarchy of grids,
  curing every wavelength on the scale where it is local; it beats even optimal SOR and
  is the modern fast Poisson solver.
- **Conjugate gradient.** The sparse system $A\boldsymbol\varphi=\mathbf b$ is symmetric
  positive-definite, so conjugate gradient (forward link to
  [§0.4](../00-foundations/linear-systems.ipynb)/[§0.5](../00-foundations/eigenvalues-svd.ipynb))
  solves it in far
  fewer steps than relaxation, especially with a preconditioner.
- **Other boundary conditions.** Neumann (fixed *normal derivative*, e.g. an insulated
  wall) and mixed conditions need only a tweak to the stencil at the boundary; the
  finite-element method handles complicated geometries with unstructured meshes.
- **Three dimensions.** Everything here generalizes by adding a third index and two more
  neighbours to the stencil; only the visualization gets harder.
- **More images.** Two parallel grounded planes need an infinite train of images; a
  point charge and a dielectric interface, a pair of fractional images. The method is a
  small art of its own.

## References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()